<table align="left">
  <td>
    <a target="_blank" href="https://drive.google.com/file/d/12qWz54NmbKlX4tLvSdVV1t-hZpPXBuGe/view?usp=sharing"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

### 데이터 출처
- 서울시에서 제공하는 수도권 생활이동 데이터를 활용하였다. 
- https://data.seoul.go.kr/dataVisual/seoul/capitalRegionLivingMigration.do
- 컬럼의 명세는 아래와 같다. 행정동 단위의 O-D 통행량 및 통행시간 정보를 제공해주고 있다. 더 상세하게는 내/외국인 구분, 국적, 이동목적과 같은 정보도 제공을 해준다.


| 순번 | 영문 컬럼명 | 컬럼 설명 | NULL 여부 | NULL 대체값 | 형식 | 규칙 | 데이터 허용범위 | 비고 |
|------|------------|-----------|-----------|-------------|------|------|-----------------|------|
| 1 | O_ADMDONG_CD | 출발 행정동 | X | - | STRING | - | - | 행안부 8자리 코드체계 |
| 2 | D_ADMDONG_CD | 도착 행정동 | X | - | STRING | - | - | 행안부 8자리 코드체계 |
| 3 | ST_TIME_CD | 출발 시간 | X | - | STRING | - | - | 7~9시/17시~19시는 20분단위, 그 외 1시간 단위 |
| 4 | FNS_TIME_CD | 도착 시간 | X | - | STRING | - | - | 7~9시/17시~19시는 20분단위, 그 외 1시간 단위 |
| 5 | IN_FORN_DIV_NM | 내/외국인 구분 | X | - | STRING | - | - | 내국인, 단기외국인, 장기외국인 |
| 6 | FORN_CITIZ_NM | 국적 | X | - | STRING | - | - | 　 |
| 7 | MOVE_PURPOSE | 이동 목적 | X | - | STRING | - | - | 1: 출근, 2 : 등교, 3: 귀가, 4: 쇼핑, 5: 관광, 6: 병원, 7: 기타 |
| 8 | MOVE_DIST | 평균 이동 거리(m) | X | - | DOUBLE | - | - | 　 |
| 9 | MOVE_TIME | 평균 이동 시간(분) | X | - | DOUBLE | - | - | 　 |
| 10 | CNT | 이동인구 수 | X | - | DOUBLE | (18,2) | - | 　 |
| 11 | ETL_YMD | 기준 년월 일 | X | - | STRING | yyyyMMdd | 데이터 기준 당일 | - |

데이터셋 명: 출도착 행정동별 이동 목적 데이터  ·
DB 명: PURPOSE_ADMDONG3

In [ ]:
data = pd.read_csv("./data/PURPOSE_ADMDONG_3_20240327.csv", encoding = 'cp949')

In [ ]:
data_grouped = data.groupby(['O_ADMDONG_CD', 'D_ADMDONG_CD','ST_TIME_CD']).agg({
    'CNT': 'sum', 
    'MOVE_DIST': 'mean',
    'MOVE_TIME':'mean'}).reset_index()

In [ ]:
data_grouped.shape

In [ ]:
data_grouped.to_parquet("./data/PURPOSE_ADMDONG_20240327.parquet")

In [ ]:
import gdrivedataset

In [ ]:
import pandas as pd
from gdrivedataset import loader
import os

def load_parquet_from_gdrive(file_id, folder="./data"):
    loader.load_from_google_drive(file_id, folder)
    file_path = os.path.join(folder, os.listdir(folder)[5])
    print(file_path)
    df = pd.read_parquet(file_path)
    return df

In [ ]:
from gdrivedataset import loader

file_id = "1teftc2KgWrG0Jn2SyyY3pipkBugmvnex"
folder = "./data_tmp"
loader.load_from_google_drive(file_id, folder)

In [ ]:
df

In [ ]:
# 함수 사용 예시
file_id = "22"
df = load_parquet_from_gdrive(file_id)

In [ ]:
df

### 1. 시간단위로 나눠져 있는 것을 일단위로 합침

In [ ]:
data_grouped = data.groupby(['O_ADMDONG_CD', 'D_ADMDONG_CD']).agg({
    'CNT': 'sum', 
    'MOVE_DIST': 'mean',
    'MOVE_TIME':'mean'}).reset_index()

# , 'MOVE_DIST', 'MOVE_TIME'
# # O_ADMDONG_CD와 D_ADMDONG_CD의 값이 같은 행 제거
# data_grouped = data_grouped[data_grouped['O_ADMDONG_CD'] != data_grouped['D_ADMDONG_CD']]


In [ ]:
data_grouped

# 'MOVE_DIST', 'MOVE_TIME' 데이터를 남기는 방법을 강구해봅시다.
#  평균 연산을 해서 남겨봅시다. 

### 2. 출발지의 통행발생량, 도착지의 통행도착량을 계산 
- 실제로는 인구 데이터를 활용하여 Trip Generation 모델을 구축해야 하지만, 생략
- 대체값으로 통행발생량, 도착량 값을 사용

In [ ]:
# 통행발생량 (O에서의 총 통행량) 계산
origin_flow = data_grouped.groupby('O_ADMDONG_CD')['CNT'].sum().reset_index().rename(columns={'CNT': 'O_FLOW'})

# 통행도착량 (D에서의 총 통행량) 계산
destination_flow = data_grouped.groupby('D_ADMDONG_CD')['CNT'].sum().reset_index().rename(columns={'CNT': 'D_FLOW'})

# 원본 데이터에 통행발생량 및 도착량을 매칭
data_grouped = data_grouped.merge(origin_flow, how='left', left_on='O_ADMDONG_CD', right_on='O_ADMDONG_CD')
data_grouped = data_grouped.merge(destination_flow, how='left', left_on='D_ADMDONG_CD', right_on='D_ADMDONG_CD')

In [ ]:
data_grouped

### 3. Trip Distribution Model 구축

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 모델 입력 변수와 타겟 변수 정의
X = data_grouped[['O_FLOW', 'D_FLOW', 'MOVE_DIST']]
X.loc[:, 'MOVE_DIST'] = X.loc[:, 'MOVE_DIST'] / 1000
y = data_grouped['CNT'] * 30

In [ ]:
# 회귀 모델 학습
model = LinearRegression()
model.fit(X, y)

In [ ]:
# 학습된 모델의 계수 (coefficients)
coefficients = model.coef_

# 학습된 모델의 절편 (intercept)
intercept = model.intercept_

# 모델 요약 정보 출력
print("모델 요약 정보:")
print("==========")
print(f"절편 (intercept): {intercept:.2f}")
print("계수 (coefficients):")
for i in range(len(X.columns)):
    print(f"- {X.columns[i]}: {coefficients[i]:.2f}")
print("==========")

중력 모델의 회귀 방정식은 다음과 같음  
$\text{CNT} = \beta_0 + \beta_1 \cdot \text{O\_FLOW} + \beta_2 \cdot \text{D\_FLOW} + \beta_3 \cdot \text{MOVE\_DIST}$  

In [ ]:
# 예측값 계산
y_pred = model.predict(X)

# RMSE 계산
rmse = np.sqrt(mean_squared_error(y, y_pred))

# MAPE 계산
mape = mean_absolute_percentage_error(y, y_pred)

# 결과 출력
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")


In [ ]:
y.describe()

In [ ]:
# 실측값 vs 예측값 산점도
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', linewidth=2)
plt.xlabel('Actual CNT')
plt.ylabel('Predicted CNT')
plt.title('Actual vs Predicted CNT')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual CNT (log scale)')
plt.ylabel('Predicted CNT (log scale)')
plt.title('Actual vs Predicted CNT (Log-Log Scale)')
plt.show()

- 각 변수에 Log 변환을 하는 방법도 있음
- 입력값에 1을 더한 후 자연로그를 계산하는 함수. x가 매우 작을 때 유용

In [ ]:

# 모델 입력 변수와 타겟 변수 정의
X = data_grouped[['O_FLOW', 'D_FLOW', 'MOVE_DIST']].copy()
X.loc[:, 'MOVE_DIST'] = X.loc[:, 'MOVE_DIST'] / 1000  # 거리 단위를 km로 변환

# 로그 변환 적용
X.loc[:, 'LOG_O_FLOW'] = np.log1p(X['O_FLOW'])
X.loc[:, 'LOG_D_FLOW'] = np.log1p(X['D_FLOW'])
X.loc[:, 'LOG_DIST'] = np.log1p(X['MOVE_DIST'])

y = np.log1p(data_grouped['CNT'] * 365)  # 연간 통행량 예측

# 선택된 특성으로 모델 입력 변수 재정의
X = X[['LOG_O_FLOW', 'LOG_D_FLOW', 'LOG_DIST']]

In [ ]:
# 회귀 모델 학습
model = LinearRegression()
model.fit(X, y)

# 예측값 계산 (로그 스케일)
y_pred_log = model.predict(X)

# 로그 변환을 풀어 실제 값으로 변환
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y)

In [ ]:
# RMSE 계산
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

# MAPE 계산
mape = mean_absolute_percentage_error(y_actual, y_pred)

# 결과 출력
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")


In [ ]:
# 실측값 vs 예측값 산점도
plt.figure(figsize=(10, 6))
plt.scatter(y_actual, y_pred, alpha=0.5)
plt.plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], '--r', linewidth=2)
plt.xlabel('Actual CNT')
plt.ylabel('Predicted CNT')
plt.title('Actual vs Predicted CNT')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_actual, y_pred, alpha=0.5)
plt.plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], '--r', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual CNT (log scale)')
plt.ylabel('Predicted CNT (log scale)')
plt.title('Actual vs Predicted CNT (Log-Log Scale)')
plt.show()

In [ ]:
# 실제 통행량을 합산했을 때는 문제가 큰것을 확인할 수 있음
y_pred.sum(),y_actual.sum()

- O와 D의 고유 ID에 대해 더미 변수를 추가하여 모델의 성능을 향상시킬 수 있음
- 극단적인 방법이나 최근 머신러닝 기술 발달과 함께 많이 쓰임

### Assignment (중요)

- 수도권 생활인구 데이터를 사용해서 O-D를 시각화 해봅시다
- 위에서 구축한 모델의 예측값과 실측값을 공간상에서 시각화해보고, 어떤 지역에서 차이가 두드러지는지를 분석해 봅시다.
- 행정동 경계 데이터는 아래 링크 데이터를 활용하세요
    - https://drive.google.com/file/d/1QRpeLj-_WNdMnqoyhvfzpNyi7iPF0O_f/view?usp=sharing
- 일전에 인구데이터로 실습했던 코드를 적극 활용하면 도움이 될 겁니다.
    - https://drive.google.com/file/d/1Qsn5srtKN1as2lEiDOVVn51lKNiaLKfD/view?usp=sharing

### 추가분석 1 - 머신러닝 모델의 사용

머신러닝의 경우 단기간에 습득하기에는 시간이 많이 걸립니다.  
본 실습에서는 이론보다는 간단한 실습을 통해 머신러닝 모델의 성능이 전통적인 모델과 비교했을 때 어떤지만 살펴보도록 합시다.  
원래는 학습(Train) 데이터와 검증(Valid) 데이터를 나눠야 하지만 본 실습에서는 생략합니다.  
본 실습에서는 LightGBM 모델을 사용합니다.  

In [ ]:
import lightgbm as lgb

In [ ]:
# 훈련 데이터와 테스트 데이터 분할
X = data_grouped[["O_ADMDONG_CD", "D_ADMDONG_CD", "MOVE_DIST", "MOVE_TIME", "O_FLOW", "D_FLOW"]]
y = data_grouped["CNT"]

In [ ]:
# LightGBM 모델 생성
model = lgb.LGBMRegressor()

# 모델 학습
model.fit(X, y)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [ ]:
# 테스트 데이터 예측
y_pred = model.predict(X)

In [ ]:
# 예측값이 0보다 작을 때는 0으로 매핑
y_pred[y_pred < 0] = 0

In [ ]:
# RMSE 계산
rmse = np.sqrt(mean_squared_error(y, y_pred))

# MAPE 계산
mape = mean_absolute_percentage_error(y, y_pred)

# 결과 출력
print(f"RMSE: {rmse}")
print(f"MAPE: {mape}")

실측값과 예측값의 비교

In [ ]:
# 실측값 vs 예측값 산점도
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', linewidth=2)
plt.xlabel('Actual CNT')
plt.ylabel('Predicted CNT')
plt.title('Actual vs Predicted CNT')
plt.show()

실측값과 예측값의 비교(log-scale)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual CNT (log scale)')
plt.ylabel('Predicted CNT (log scale)')
plt.title('Actual vs Predicted CNT (Log-Log Scale)')
plt.show()

### 추가분석 2 - O-D 데이터 시각화

단순히 시각화만 하지 말고, 실제값과 예측값을 비교해봅시다  

In [ ]:
data_grouped

실측값과 예측값의 차이 비교를 위해, 머신러닝 모델로 예측한 값을 새로운 컬럼으로 생성  
그 후에 실측값과 예측값의 차이를 계산합시다.  

In [ ]:
data_grouped['y_pred'] = y_pred
data_grouped['diff'] = np.abs(data_grouped['CNT'] - data_grouped['y_pred'])

위의 데이터는 현재 출발지, 목적지 읍면동 코드는 존재하지만 기하정보 데이터가 존재하지 않습니다. 
기하정보 데이터를 붙여봅시다. 

In [ ]:
# 파일 경로 설정
file_path = './data/HangJeongDong_ver20230101.geojson'

# 오류를 해결하기 위해 다른 방법으로 파일을 다시 불러옴
geo_data = gpd.read_file(file_path, driver='GeoJSON')

# 데이터의 기본 정보 및 첫 몇 줄 확인
geo_data.info(), geo_data.head()

읍면동 코드의 자릿수가 다르므로 100을 나눠서 동일하게 맞춰줍시다.

In [ ]:
geo_data['adm_cd2'] = pd.to_numeric(geo_data['adm_cd2'])
geo_data['adm_cd2'] = (geo_data['adm_cd2'] / 100).astype(int)

위경도를 나타내는 컬럼이 없으므로, Polygon의 중심점을 뽑아서 만들어봅시다. 

In [ ]:
# Centroid 추출
geo_data["centroid"] = geo_data.geometry.centroid

# 새로운 컬럼 생성
geo_data["lon"] = geo_data["centroid"].x
geo_data["lat"] = geo_data["centroid"].y

In [ ]:
geo_data_filtered = geo_data[['adm_nm','adm_cd2','lon','lat','sido']]

In [ ]:
geo_data_filtered

전체 데이터 중, `geo_data`에 코드가 없을 경우 해당 행을 제거함

In [ ]:
data_grouped_filtered = data_grouped.merge(geo_data_filtered, how='inner', left_on='O_ADMDONG_CD', right_on='adm_cd2')

In [ ]:
data_grouped.shape, data_grouped_filtered.shape

In [ ]:
data_grouped_filtered = data_grouped_filtered.merge(geo_data_filtered, how='inner', left_on='D_ADMDONG_CD', right_on='adm_cd2')

In [ ]:
data_grouped.shape, data_grouped_filtered.shape
# 총 1017973행 중, 881847행만 남음

In [ ]:
data_grouped

시각화를 위한 최종 데이터 저장

In [ ]:
import os

In [ ]:
path = './result/'
geo_data_filtered.to_parquet(os.path.join(path + 'locations_seoul_metropolitan.parquet'))
data_grouped_filtered.to_parquet(os.path.join(path + 'flows_seoul_metropolitan.parquet'))

##### flowmap 사이트를 통해 Render
https://www.flowmap.city/

- 아래와 같은 결과가 도출되면 됨  
https://app.flowmap.city/public/27bcc2a7-03ff-44c7-9e6b-99c9382915d4

### Experiment

#### 연습 1
- 현재 데이터는 수도권에서 타지역으로 나가는 통행까지도 포함하고 있습니다. 분석 대상지가 수도권 안이라고 하였을 때, 필요없는 통행을 제거한 후, 다시 시각화를 해 봅시다.
- 힌트: sido 코드 기준 11:서울; 23:인천, 경기도:31 입니다.
- 시도 코드표: https://sgis.kostat.go.kr/developer/html/openApi/api/dataCode/SidoCode.html


#### 연습 2
- `diff` 컬럼을 출발지 기준으로 합산해봅시다. 출발지 기준, 가장 많은 오차를 발생시키는 동은 어디인가요?